# Lesson 6B: Naive Bayes Practical

<a name="introduction"></a>
## Introduction

Lesson 6a derived Bayes' theorem, the conditional independence assumption, and
three Naive Bayes variants from first principles, including a from-scratch
Gaussian Naive Bayes implementation. This lesson applies the Multinomial variant
to a real text classification problem, where the "naive" independence assumption
is most commonly deployed in practice.

Text data introduces two practical concerns theory alone does not resolve:

1. **Representation**: raw word counts favor common but uninformative words
   (TF-IDF reweights them by how discriminative they are across documents)
2. **Sparsity**: any single document uses a tiny fraction of the full vocabulary,
   so most word counts are zero — this is exactly the zero-frequency problem
   Laplace smoothing was built to solve

In this lesson, we'll:
1. Load a real multi-class text classification dataset (20 Newsgroups subset)
2. Derive and implement TF-IDF vectorization
3. Implement Multinomial Naive Bayes from scratch (extending the 6a theory) and confirm it matches scikit-learn
4. Demonstrate empirically why Laplace smoothing is necessary for text
5. Show why Gaussian Naive Bayes is a poor fit for sparse text features
6. Analyze performance and inspect the most informative features per class
7. Perform error analysis on misclassified documents


## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [Dataset: 20 Newsgroups Subset](#dataset-20-newsgroups-subset)
4. [TF-IDF Vectorization](#tf-idf-vectorization)
   - [Term Frequency](#term-frequency)
   - [Inverse Document Frequency](#inverse-document-frequency)
   - [Combining TF and IDF](#combining-tf-and-idf)
5. [From-Scratch Multinomial Naive Bayes](#from-scratch-multinomial-naive-bayes)
   - [Implementation](#implementation)
   - [Comparison to Scikit-learn](#comparison-to-scikit-learn)
6. [The Zero-Frequency Problem in Practice](#the-zero-frequency-problem-in-practice)
   - [Laplace Smoothing: Empirical Effect](#laplace-smoothing-empirical-effect)
7. [Why Gaussian Naive Bayes Fails on Text](#why-gaussian-naive-bayes-fails-on-text)
8. [Performance Analysis](#performance-analysis)
   - [Confusion Matrix](#confusion-matrix)
   - [Most Informative Features](#most-informative-features)
9. [Error Analysis](#error-analysis)
10. [Conclusion](#conclusion)
    - [Key Insights](#key-insights-2)
    - [Further Reading](#further-reading-2)


<a name="required-libraries"></a>
## Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


<a name="dataset-20-newsgroups-subset"></a>
## Dataset: 20 Newsgroups Subset

We use four categories from the 20 Newsgroups dataset — a standard text
classification benchmark of Usenet posts. Restricting to four well-separated
topics keeps the problem multi-class (unlike the binary SVM/spam examples used
elsewhere in this course) while keeping vocabulary and training time manageable.


In [ ]:
categories = ['alt.atheism', 'comp.graphics', 'sci.med', 'rec.sport.baseball']

train_data = fetch_20newsgroups(
    subset='train', categories=categories, remove=('headers', 'footers', 'quotes'),
    random_state=42
)
test_data = fetch_20newsgroups(
    subset='test', categories=categories, remove=('headers', 'footers', 'quotes'),
    random_state=42
)

print("\n" + "="*70)
print("20 NEWSGROUPS SUBSET")
print("="*70)
print(f"\nCategories: {categories}")
print(f"Training documents: {len(train_data.data)}")
print(f"Test documents: {len(test_data.data)}")
print(f"\nClass distribution (train):")
for i, cat in enumerate(train_data.target_names):
    count = np.sum(train_data.target == i)
    print(f"  {cat}: {count} documents")

print(f"\nSample document (class '{train_data.target_names[train_data.target[0]]}'):")
print(train_data.data[0][:300])


<a name="tf-idf-vectorization"></a>
## TF-IDF Vectorization

<a name="term-frequency"></a>
### Term Frequency

Raw word counts treat every word as equally informative, but common words
("the", "is") appear in nearly every document regardless of topic. **Term
frequency** for word $t$ in document $d$ is the (often normalized) count:

$$\text{tf}(t, d) = \frac{\text{count of } t \text{ in } d}{\text{total words in } d}$$

<a name="inverse-document-frequency"></a>
### Inverse Document Frequency

**Inverse document frequency** downweights words that appear in many documents
across the corpus, since a word present everywhere carries little information
about any specific document's topic:

$$\text{idf}(t) = \log\left(\frac{N}{1 + \text{df}(t)}\right) + 1$$

where $N$ is the total number of documents and $\text{df}(t)$ is the number of
documents containing term $t$. The $+1$ in the denominator is Laplace-style
smoothing to avoid division by zero for terms present in every document; the
trailing $+1$ prevents terms in all documents from being weighted exactly zero.

<a name="combining-tf-and-idf"></a>
### Combining TF and IDF

The final weight is the product:

$$\text{tf-idf}(t, d) = \text{tf}(t, d) \times \text{idf}(t)$$

A word gets high tf-idf weight when it appears frequently in a specific document
($\text{tf}$ is high) but rarely across the whole corpus ($\text{idf}$ is
high) — exactly the words that are discriminative for that document's topic.
scikit-learn's `TfidfVectorizer` additionally L2-normalizes each document's
vector so document length does not bias the comparison.


In [ ]:
# Fit TF-IDF on the training set, transform both train and test
tfidf = TfidfVectorizer(max_features=5000, stop_words='english', min_df=2)
X_train_tfidf = tfidf.fit_transform(train_data.data)
X_test_tfidf = tfidf.transform(test_data.data)

y_train = train_data.target
y_test = test_data.target

print("\n" + "="*70)
print("TF-IDF VECTORIZATION")
print("="*70)
print(f"\nVocabulary size: {len(tfidf.vocabulary_)}")
print(f"Training matrix shape: {X_train_tfidf.shape} (sparse)")
print(f"Test matrix shape: {X_test_tfidf.shape} (sparse)")
print(f"Sparsity: {1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]):.4%} zeros")

# Show tf-idf weights for a sample document's top terms
feature_names = np.array(tfidf.get_feature_names_out())
sample_vec = X_train_tfidf[0].toarray().flatten()
top_indices = sample_vec.argsort()[-10:][::-1]
print(f"\nTop TF-IDF terms in sample document (class '{train_data.target_names[y_train[0]]}'):")
for idx in top_indices:
    if sample_vec[idx] > 0:
        print(f"  {feature_names[idx]:<20} tf-idf = {sample_vec[idx]:.4f}")


<a name="from-scratch-tfidf"></a>
### From-Scratch TF-IDF in NumPy

To confirm the formulas above are correctly understood (not just correctly
called via a library), we implement raw-count TF-IDF directly in NumPy and
compare it to scikit-learn's `TfidfVectorizer` on the same small vocabulary.
scikit-learn's default smooths and L2-normalizes; we replicate both so the
two outputs are directly comparable.


In [ ]:
def tfidf_from_scratch(count_matrix):
    """
    Compute TF-IDF from a raw document-term count matrix (dense NumPy array),
    matching scikit-learn's smooth-idf + L2-normalize defaults.

    tf(t, d)  = raw count of term t in document d (scikit-learn's default: no
                length normalization at the tf stage)
    idf(t)    = log((1 + N) / (1 + df(t))) + 1        [smooth idf]
    tf-idf    = tf * idf, then each document row L2-normalized
    """
    n_docs, n_terms = count_matrix.shape

    df = np.count_nonzero(count_matrix, axis=0)                 # document frequency per term
    idf = np.log((1 + n_docs) / (1 + df)) + 1                    # smoothed idf

    tfidf = count_matrix * idf                                   # broadcast idf across documents
    row_norms = np.linalg.norm(tfidf, axis=1, keepdims=True)
    row_norms[row_norms == 0] = 1  # avoid division by zero for empty documents
    return tfidf / row_norms


# IDF must be computed over the FULL training corpus (matching how the
# scikit-learn vectorizer was fit above) -- computing it from a small
# subset would give different document frequencies and a wrong result.
count_vec_full = CountVectorizer(vocabulary=tfidf.vocabulary_)
count_full = count_vec_full.fit_transform(train_data.data).toarray()
tfidf_scratch_full = tfidf_from_scratch(count_full)
tfidf_scratch = tfidf_scratch_full[:50]
tfidf_sklearn_subset = tfidf.transform(train_data.data[:50]).toarray()

max_diff = np.abs(tfidf_scratch - tfidf_sklearn_subset).max()

print("\n" + "="*70)
print("FROM-SCRATCH vs SCIKIT-LEARN TF-IDF")
print("="*70)
print(f"\nIDF fit on {count_full.shape[0]} training documents; compared on the first "
      f"{tfidf_scratch.shape[0]} documents, {tfidf_scratch.shape[1]} terms")
print(f"Max absolute difference: {max_diff:.2e}")
print("\nThe two outputs match to within floating-point precision, confirming")
print("the NumPy implementation reproduces scikit-learn's smoothed, L2-normalized")
print("TF-IDF weighting exactly.")


<a name="from-scratch-multinomial-naive-bayes"></a>
## From-Scratch Multinomial Naive Bayes

<a name="implementation"></a>
### Implementation

Lesson 6a derived the Multinomial NB parameter estimate with Laplace smoothing:

$$\hat{\theta}_{i,y} = \frac{\alpha + \sum_{n: y_n = y} x_{n,i}}{\alpha d + \sum_{j=1}^{d} \sum_{n: y_n = y} x_{n,j}}$$

We implement this directly. Since scikit-learn's `TfidfVectorizer` produces
non-negative real-valued weights (not integer counts), the multinomial model
is applied to these weights directly — this is the standard practice for TF-IDF
features and is what scikit-learn's `MultinomialNB` does as well.


In [ ]:
from scipy.sparse import issparse

class MultinomialNaiveBayes:
    """
    Multinomial Naive Bayes with Laplace (additive) smoothing.

    theta_hat[i, y] = (alpha + sum of feature i over class-y documents)
                      / (alpha * d + sum of all features over class-y documents)
    Classification: argmax_y [ log P(y) + sum_i x_i * log theta_hat[i, y] ]
    """

    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes_ = None
        self.class_log_prior_ = None
        self.feature_log_prob_ = None

    def fit(self, X, y):
        if issparse(X):
            X = X.toarray()
        n_samples, n_features = X.shape
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)

        self.class_log_prior_ = np.zeros(n_classes)
        self.feature_log_prob_ = np.zeros((n_classes, n_features))

        for idx, c in enumerate(self.classes_):
            X_c = X[y == c]
            self.class_log_prior_[idx] = np.log(X_c.shape[0] / n_samples)

            feature_counts = X_c.sum(axis=0)          # sum_{n: y_n=y} x_{n,i}, per feature
            total_count = feature_counts.sum()         # sum_j sum_{n: y_n=y} x_{n,j}

            smoothed = (self.alpha + feature_counts) / (self.alpha * n_features + total_count)
            self.feature_log_prob_[idx, :] = np.log(smoothed)

        return self

    def _joint_log_likelihood(self, X):
        if issparse(X):
            X = X.toarray()
        # log P(y) + sum_i x_i * log theta_iy  ==  X @ feature_log_prob_^T + class_log_prior_
        return X @ self.feature_log_prob_.T + self.class_log_prior_

    def predict(self, X):
        jll = self._joint_log_likelihood(X)
        return self.classes_[np.argmax(jll, axis=1)]


print("\n" + "="*70)
print("FROM-SCRATCH MULTINOMIAL NAIVE BAYES")
print("="*70)
print("\nTraining: for each class, Laplace-smoothed relative frequency per feature")
print("Prediction: argmax over log P(y) + sum_i x_i * log theta_hat[i,y]")


<a name="comparison-to-scikit-learn"></a>
### Comparison to Scikit-learn

In [ ]:
# Train from-scratch and scikit-learn Multinomial NB, compare directly
mnb_scratch = MultinomialNaiveBayes(alpha=1.0)
mnb_scratch.fit(X_train_tfidf, y_train)

y_pred_scratch = mnb_scratch.predict(X_test_tfidf)
scratch_accuracy = accuracy_score(y_test, y_pred_scratch)

mnb_sklearn = MultinomialNB(alpha=1.0)
mnb_sklearn.fit(X_train_tfidf, y_train)
y_pred_sklearn = mnb_sklearn.predict(X_test_tfidf)
sklearn_accuracy = accuracy_score(y_test, y_pred_sklearn)

print("\n" + "="*70)
print("FROM-SCRATCH vs SCIKIT-LEARN MULTINOMIALNB")
print("="*70)
print(f"\n{'Model':<30}{'Test Accuracy':<18}")
print("-"*70)
print(f"{'From-scratch MultinomialNB':<30}{scratch_accuracy:<18.4f}")
print(f"{'scikit-learn MultinomialNB':<30}{sklearn_accuracy:<18.4f}")

agreement = np.mean(y_pred_scratch == y_pred_sklearn)
max_param_diff = np.abs(mnb_scratch.feature_log_prob_ - mnb_sklearn.feature_log_prob_).max()
print(f"\nAgreement between predictions: {agreement:.4f}")
print(f"Max difference in fitted log-probabilities: {max_param_diff:.2e}")
print("\nBoth compute the same closed-form Laplace-smoothed estimate, so they")
print("match to within floating-point precision, exactly as with Gaussian NB in 6a.")


<a name="the-zero-frequency-problem-in-practice"></a>
## The Zero-Frequency Problem in Practice

<a name="laplace-smoothing-empirical-effect"></a>
### Laplace Smoothing: Empirical Effect

Lesson 6a explained why omitting Laplace smoothing ($\alpha=0$) causes any
document containing a word absent from a class's training vocabulary to receive
zero posterior probability for that class. We now measure this directly: how
many test documents contain at least one term that never appeared in a given
class's training documents?


In [ ]:
# Count vectorizer (raw counts) to make the zero-frequency problem concrete
count_vec = CountVectorizer(max_features=5000, stop_words='english', min_df=2, vocabulary=tfidf.vocabulary_)
X_train_counts = count_vec.fit_transform(train_data.data)
X_test_counts = count_vec.transform(test_data.data)

print("\n" + "="*70)
print("HOW OFTEN DOES THE ZERO-FREQUENCY PROBLEM OCCUR?")
print("="*70)

for idx, c in enumerate(np.unique(y_train)):
    train_class_counts = np.asarray(X_train_counts[y_train == c].sum(axis=0)).flatten()
    words_never_seen_in_class = (train_class_counts == 0)

    # For each test document, does it contain any word never seen in this class's training data?
    test_doc_words_present = (X_test_counts.toarray() > 0)
    affected_docs = np.any(test_doc_words_present[:, words_never_seen_in_class], axis=1)

    print(f"\nClass '{train_data.target_names[c]}':")
    print(f"  Vocabulary words never seen in this class's training docs: "
          f"{words_never_seen_in_class.sum()} / {len(words_never_seen_in_class)}")
    print(f"  Test documents affected (contain such a word): "
          f"{affected_docs.sum()} / {len(affected_docs)} ({affected_docs.mean():.1%})")

print("\nWithout smoothing, every affected test document would receive exactly")
print("zero posterior probability for that class -- regardless of how well every")
print("other word in the document matches the class. This is why alpha=0 is never")
print("used in practice for text classification.")


In [ ]:
# Compare accuracy across a range of alpha values, including alpha near 0
alphas = [1e-5, 0.001, 0.01, 0.1, 1.0, 5.0, 20.0]
accuracies = []

for alpha in alphas:
    mnb = MultinomialNB(alpha=alpha)
    mnb.fit(X_train_tfidf, y_train)
    acc = accuracy_score(y_test, mnb.predict(X_test_tfidf))
    accuracies.append(acc)
    print(f"alpha={alpha:<10.5f} test accuracy = {acc:.4f}")

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(alphas, accuracies, 'o-', linewidth=2, markersize=8, color='steelblue')
ax.set_xscale('log')
ax.set_xlabel('Laplace smoothing parameter (alpha, log scale)')
ax.set_ylabel('Test accuracy')
ax.set_title('Effect of Laplace Smoothing on Classification Accuracy')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nVery small alpha (near-zero smoothing) hurts accuracy because a single")
print("unseen word can dominate the log-posterior; very large alpha over-smooths,")
print("pushing all class probabilities toward uniform and erasing the signal in")
print("the word counts. The useful range sits in between.")


<a name="why-gaussian-naive-bayes-fails-on-text"></a>
## Why Gaussian Naive Bayes Fails on Text

Lesson 6a's Gaussian Naive Bayes assumes each feature is continuous and
approximately normally distributed within a class. TF-IDF features violate
this badly: the vast majority of entries are exactly zero (a word either
appears in a document or it does not), so the empirical distribution of any
single TF-IDF feature is a spike at zero with a long right tail — nothing like
a bell curve. We demonstrate this by applying `GaussianNB` to the same TF-IDF
features used above.


In [ ]:
# Apply GaussianNB to the same TF-IDF features (dense conversion required)
X_train_dense = X_train_tfidf.toarray()
X_test_dense = X_test_tfidf.toarray()

gnb_text = GaussianNB()
gnb_text.fit(X_train_dense, y_train)
y_pred_gnb = gnb_text.predict(X_test_dense)
gnb_accuracy = accuracy_score(y_test, y_pred_gnb)

print("\n" + "="*70)
print("GAUSSIANNB vs MULTINOMIALNB ON THE SAME TF-IDF FEATURES")
print("="*70)
print(f"\n{'Model':<30}{'Test Accuracy':<18}")
print("-"*70)
print(f"{'MultinomialNB (appropriate)':<30}{sklearn_accuracy:<18.4f}")
print(f"{'GaussianNB (inappropriate)':<30}{gnb_accuracy:<18.4f}")

# Show why: fraction of zero entries per feature (sparsity), which breaks the
# "continuous, normally distributed" assumption Gaussian NB requires
zero_fraction = (X_train_dense == 0).mean(axis=0)
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.hist(zero_fraction, bins=30, color='indianred', edgecolor='k', alpha=0.7)
ax.set_title('Fraction of Documents Where Each Word Has Zero TF-IDF Weight')
ax.set_xlabel('Fraction of zero entries (sparsity) per feature')
ax.set_ylabel('Number of features')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nMedian feature sparsity: {np.median(zero_fraction):.1%} of documents have")
print("zero weight for the median word. A feature that is mostly zero with an")
print("occasional positive spike is not well modeled by a Gaussian -- Gaussian NB")
print("wastes parameters fitting a bell curve to what is really a point mass at")
print("zero plus a sparse tail, which is exactly why the accuracy above is lower.")


<a name="performance-analysis"></a>
## Performance Analysis

<a name="confusion-matrix"></a>
### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_sklearn)

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=train_data.target_names, yticklabels=train_data.target_names)
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title('Confusion Matrix: MultinomialNB on 20 Newsgroups Subset')
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("CLASSIFICATION REPORT")
print("="*70)
print(classification_report(y_test, y_pred_sklearn, target_names=train_data.target_names))

macro_f1 = f1_score(y_test, y_pred_sklearn, average='macro')
print(f"Macro F1: {macro_f1:.4f}")


<a name="most-informative-features"></a>
### Most Informative Features

In [ ]:
# For each class, find the words with the highest log-probability
# (i.e. the words that most strongly indicate that class)
n_top = 10
feature_names = np.array(tfidf.get_feature_names_out())

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, class_name in enumerate(train_data.target_names):
    log_probs = mnb_sklearn.feature_log_prob_[idx]
    top_indices = np.argsort(log_probs)[-n_top:]
    top_words = feature_names[top_indices]
    top_values = log_probs[top_indices]

    ax = axes[idx]
    ax.barh(range(n_top), top_values, color='steelblue')
    ax.set_yticks(range(n_top))
    ax.set_yticklabels(top_words)
    ax.set_xlabel('log P(word | class)')
    ax.set_title(f"Most Informative Words: '{class_name}'")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nThese are the words with the highest fitted probability within each")
print("class -- they align with each newsgroup's actual subject matter, which")
print("is a useful sanity check that the model learned something meaningful")
print("rather than an artifact of the vectorization.")


<a name="error-analysis"></a>
## Error Analysis

In [ ]:
# Inspect misclassified documents
misclassified = np.where(y_pred_sklearn != y_test)[0]

print("\n" + "="*70)
print(f"ERROR ANALYSIS: {len(misclassified)} / {len(y_test)} test documents misclassified "
      f"({len(misclassified)/len(y_test):.1%})")
print("="*70)

# Confusion pairs: which classes get confused with which
confused_pairs = {}
for i in misclassified:
    pair = (train_data.target_names[y_test[i]], train_data.target_names[y_pred_sklearn[i]])
    confused_pairs[pair] = confused_pairs.get(pair, 0) + 1

print("\nMost common misclassification pairs (true -> predicted):")
for pair, count in sorted(confused_pairs.items(), key=lambda kv: -kv[1])[:5]:
    print(f"  {pair[0]:<20} -> {pair[1]:<20} : {count} documents")

# Show a couple of example misclassified documents
print("\nExample misclassified documents:")
for i in misclassified[:3]:
    true_label = train_data.target_names[y_test[i]]
    pred_label = train_data.target_names[y_pred_sklearn[i]]
    print(f"\n  True: {true_label} | Predicted: {pred_label}")
    print(f"  Text (first 200 chars): {test_data.data[i][:200].strip()}")

print("\nMost errors occur between topically related or short/ambiguous")
print("documents -- a short post with few discriminative words has little")
print("signal for TF-IDF to amplify, so the classifier falls back closer to")
print("the class priors.")


<a name="conclusion"></a>
## Conclusion

<a name="key-insights-2"></a>
### Key Insights

1. **TF-IDF** reweights raw word counts by how discriminative they are across
   the corpus, giving Naive Bayes a stronger signal than raw counts alone

2. **The from-scratch Multinomial NB** implementation matches scikit-learn
   exactly, confirming the Laplace-smoothed MLE derivation from Lesson 6a
   transfers directly to real text data

3. **Laplace smoothing is not optional for text**: a large fraction of test
   documents contain at least one word absent from a given class's training
   vocabulary, and without smoothing those documents get zero posterior
   probability for that class

4. **Gaussian Naive Bayes is a poor fit for sparse text features** — TF-IDF
   weights are mostly zero with an occasional spike, nothing like the
   continuous, unimodal distribution the Gaussian model assumes

5. **Most informative features** provide a useful sanity check that the model
   has learned genuine topic signal rather than vectorization artifacts

6. **Errors concentrate on short or topically ambiguous documents**, where
   TF-IDF has little discriminative signal to amplify


<a name="further-reading-2"></a>
### Further Reading

- scikit-learn documentation: "Working With Text Data" tutorial (`TfidfVectorizer`, `MultinomialNB`)
- scikit-learn documentation: `sklearn.feature_extraction.text` module reference
- Bishop, C. M. (2006). "Pattern Recognition and Machine Learning (PRML)", Chapter 4
- Lesson 6a: Naive Bayes Theory (Bayes' theorem, conditional independence, Laplace smoothing derivation)
